© 2026 WaLSA Team - Shahin Jafarzadeh et al.

This notebook is part of the [WaLSAlib](https://github.com/WaLSAteam/WaLSAlib) package (v1.0.0), provided under the [Apache License, Version 2.0](http://www.apache.org/licenses/LICENSE-2.0).

You may use, modify, and distribute this notebook and its contents under the terms of the license.

---

The Proof-of-Principle Hybrid Emulator generated using this notebook **is published in**:  
**Jafarzadeh, S., Jess, D. B., Stangalini, M. et al. 2026, *Frontiers in Astronomy and Space Sciences*, in prep.**,  
as part of the research topic: **[Magnetohydrodynamic Motions: Daniel K. Inouye Solar Telescope’s Window into the Dynamic Sun](https://www.frontiersin.org/research-topics/71781/magnetohydrodynamic-motions-daniel-k-inouye-solar-telescopes-window-into-the-dynamic-sun)**,     

---

**Disclaimer**:
This notebook and its code are provided "as is", without warranty of any kind, express or implied. Refer to the license for more details.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
WaLSA LineFit — Proof-of-Principle Hybrid Acceleration via Supervised Emulation
================================================================================

Overview
--------
This script implements and evaluates a lightweight supervised CNN emulator that
accelerates line-centre extraction relative to the full parametric LineFit solver,
while retaining LineFit as a targeted fallback for morphologically complex profiles.

The emulator is trained to predict *residual line-centre shifts relative to a
robust coarse centre* (the "residual-to-coarse" formulation), which reduces the
dynamic range the network must learn and decouples coarse localisation from
fine-structure refinement. LineFit outputs serve as training labels; ground truth
from the synthetic generator is used only for validation and verification.

A hybrid inference mode is applied specifically to the known stress-case line
(intermittently split-core morphology). Two gates are evaluated:
  (i)  A morphology flag triggered on split-core-like or multi-peak structures.
  (ii) An MC-dropout uncertainty estimate calibrated on a held-out calibration set.
Only morphology-flagged samples whose MC-dropout standard deviation exceeds a
calibrated threshold are routed to the LineFit fallback. For the timing
benchmark, LineFit is invoked on a small sample of fallback time steps (5 by
default) to estimate the per-step fallback cost, which is then extrapolated to
all fallback steps. This is more accurate than using the full-run average
because fallback steps are the most morphologically complex and are typically
slower than average, and more practical than timing all fallback steps (which
would take ~70 minutes for 62 steps in this configuration).

This is an explicitly limited proof-of-principle. The architecture, hyperparameters,
and hybrid logic are intentionally kept simple. The primary purpose is to demonstrate
feasibility and quantify the accuracy/speed trade-off, not to provide a production-
ready emulator. Full development (larger training sets, cross-validation, broader
morphology coverage, and instrument-specific tuning) is left for future work.

Inputs
------
  synthetic_nuv_testbed.fits
      FITS file with extensions: [0] spectra (nt, nw), WAVELENGTH, TIME,
      LAMBDA0, CENTER_TRUE.
  Files/Fig3_cache/fig3_vtimeseries__v_linefit.npy
      Cached LineFit LOS velocities, shape (nt, n_lines) or (n_lines, nt).

Outputs  (written under OUTDIR)
--------------------------------
  cnn_model.pt                  — saved model weights + metadata
  metrics.json                  — accuracy metrics (val vs LineFit and vs Truth)
  hybrid_summary.json           — fallback fraction, threshold, gating details
  timing.json                   — hybrid timing with sample-based fallback estimate
  training_curves.npz           — per-epoch train/val loss (for convergence audit)
  predictions_line6.npz         — stress-line time-series products
  val_diagnostics.npz           — full validation-set diagnostics
  plots/
      val_error_hist_cdf.pdf
      fallback_map.pdf
      rgws_line6_comparison.pdf

Dependencies
------------
  numpy, scipy, astropy, torch (>=1.12), WaLSAlib,
  matplotlib (optional), WaLSAtools (optional)

Configuration
-------------
  All configuration is via the constants in the "Configuration" section below.

Authors
-------
  WaLSA Team / Shahin Jafarzadeh (2026)
  https://WaLSA.team

License
-------
  Apache-2.0 license — see LICENSE file in the WaLSAlib repository.
  https://github.com/WaLSAteam/WaLSAlib

"""

from __future__ import annotations

import os
import json
import time
import warnings
from dataclasses import dataclass
from typing import Dict, Tuple, Any, List, Optional

import numpy as np
from astropy.io import fits
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# WaLSAlib LineFit — required for fallback timing measurement
from WaLSAlib import linefit

# ---------------------------------------------------------------------------
# Optional dependencies
# ---------------------------------------------------------------------------
try:
    import matplotlib
    matplotlib.use("Agg")   # non-interactive backend for batch/HPC use
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

try:
    from WaLSAtools import WaLSAtools, WaLSA_detrend_apod  # type: ignore
    HAVE_WALSA = True
except Exception:
    HAVE_WALSA = False


# =============================================================================
# Configuration
# =============================================================================

# --- I/O paths ---------------------------------------------------------------
FITS_PATH      = "synthetic_nuv_testbed.fits"
V_LINEFIT_NPY  = os.path.join("Files", "Fig3_cache", "fig3_vtimeseries__v_linefit.npy")
OUTDIR         = os.path.join("Files", "PoP_CNN_emulator_outputs")
PLOTDIR        = os.path.join(OUTDIR, "plots")

# --- Spectral configuration (must match the FITS testbed) --------------------
# Per-line half-window sizes in wavelength-axis index bins.
PER_LINE_HALF_WIN: List[int] = [10, 10, 10, 10, 10, 28, 18, 10, 10, 20]

# Declared close pairs for ownership-split seeding (0-based line indices).
CLOSE_PAIRS: List[Tuple[int, int]] = [(3, 4)]

# Parameters for coarse centre finder.
SEARCH_HALF_WIN: int   = 120
SMOOTH_COARSE:   int   = 3

# Index of the "stress-case" line (split-core / intermittent emission reversal).
# Morphology gating and MC-dropout are applied only to this line.
LINE_STRESS: int = 5

# --- Dataset split -----------------------------------------------------------
# Time-ordered split to prevent temporal information leakage.
# Validation uses the final (1 - TRAIN_FRAC_TIME) fraction of the time series.
# Within the training window, the last CAL_FRAC_WITHIN_TRAIN fraction is used
# for calibrating the uncertainty threshold (no gradient updates on CAL).
TRAIN_FRAC_TIME:        float = 0.80
CAL_FRAC_WITHIN_TRAIN:  float = 0.10

# Whether to standardize the residual target per line (zero-mean, unit-std).
# Strongly recommended: reduces scale differences across lines.
STANDARDIZE_TARGET_PER_LINE: bool = True

# --- Morphology detection thresholds (stress line only) ----------------------
SPLIT_PROM_FRAC:      float = 0.06   # prominence threshold relative to window range
SPLIT_VALLEY_FRAC:    float = 0.85   # valley must be below this fraction of min-peak
MULTIPEAK_HEIGHT_REL: float = 0.20   # secondary-peak height threshold (relative to max)

# --- Hybrid / uncertainty parameters -----------------------------------------
# Number of stochastic forward passes for MC-dropout uncertainty estimation.
MC_DROPOUT_PASSES: int = 30

# --- RGWS (wave diagnostic) settings — must match the main paper analysis ----
APOD:      float = 0.10
SIGLEVEL:  float = 0.95
MOTHER:    str   = "morlet"

# --- PyTorch threading -------------------------------------------------------
# Set to None to use PyTorch default. Reduce for shared compute nodes.
TORCH_NUM_THREADS: Optional[int] = 6

# --- Fixed model and training configuration (frozen PoP reference) -----------
# These hyperparameters were selected by a preliminary search and are fixed
# here for a fully reproducible paper run. FINAL_SEED should only be changed if a new
# independent run is explicitly desired.
FINAL_SEED: int = 7000
FINAL_CFG: Dict[str, Any] = {
    "lr":                  1.892149291938482e-3,
    "weight_decay":        7.61522710343447e-5,
    "dropout_p":           0.04,
    "huber_beta":          0.05,
    "emb_dim":             16,
    "batch_size":          256,
    "sig_floor_pm":        0.05,   # minimum per-line std for standardization
    "ema_decay":           0.995,
    "stress_diff_lambda":  3e-4,   # temporal smoothness regularizer strength
}
FINAL_EPOCHS:   int = 160
FINAL_PATIENCE: int = 16

# --- EMA (Exponential Moving Average) of weights ----------------------------
USE_EMA: bool = True

# Which weight snapshot to report as the final model: "best" or "ema".
PICK_WEIGHTS_TAG: str = "best"

# --- Timing benchmark --------------------------------------------------------
INFER_REPEATS: int = 400
TIMING_WARMUP: int = 25
USE_TORCHSCRIPT_TRACE_FOR_TIMING: bool = True

# --- Physical constant -------------------------------------------------------
C_KMS: float = 299792.458   # speed of light (km/s)

# =============================================================================
# LineFit fallback configuration
# These parameters match those used to generate the cached LineFit outputs
# (i.e. the training labels in fig3_vtimeseries_calc.py). Any mismatch would
# produce a timing measurement that is inconsistent with the label-generation
# run. Copied verbatim from Figure3.ipynb
# =============================================================================
_LF_NUM_ITERATIONS:   int       = 20
_LF_WINDOW_SIZE:      int       = 14
_LF_SMOOTHING_WINDOW: int       = 1
_LF_PER_LINE_WIN:     List[int] = [10, 10, 10, 10, 10, 28, 18, 10, 10, 20]
_LF_PER_LINE_MIN_HW:  List[int] = [8,  10,  8,  8,  8, 14, 10,  8,  8, 12]
_LF_PER_LINE_MAX_HW:  List[int] = [14, 13, 12, 12, 12, 36, 22, 14, 12, 24]
_LF_USE_COARSE:       bool      = True
_LF_ADAPTIVE_WIN:     bool      = True
_LF_CLOSE_PAIRS:      List[Tuple[int, int]] = [(3, 4)]
_LF_REVERSAL_LINES:   List[int] = [5]
# Suppress CSV/FITS output during the timing-only fallback run
_LF_WRITE_CSV:        bool      = False
_LF_SAVE_FITTED:      bool      = False
_LF_SILENT:           bool      = True

# Total LineFit runtime for the full 300-time-step label-generation run,
# estimated by timing _LF_TIMING_NFRAMES uniformly sampled steps
# and scaling — the same approach used here for the fallback timing.
_LINEFIT_FULL_TOTAL_S: float = 52997.0
_LF_TIMING_NFRAMES:   int   = 10   # number of frames used to estimate the above


# =============================================================================
# Reproducibility utilities
# =============================================================================

def set_seed(seed: int) -> None:
    """Set random seeds for NumPy, PyTorch CPU, and CUDA (if available)."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def nearest_index(wl_nm: np.ndarray, x_nm: float) -> int:
    """Return the index of the wavelength axis element closest to x_nm."""
    return int(np.argmin(np.abs(wl_nm - float(x_nm))))


# =============================================================================
# Spectrum normalisation
# =============================================================================

def minmax01(a: np.ndarray) -> np.ndarray:
    """
    Normalise array to [0, 1] using global min/max.

    If the array has zero range (constant or non-finite), returns an array of
    zeros with a small non-zero first element to prevent downstream division
    by zero in subsequent operations.
    """
    a = np.asarray(a, float)
    mn, mx = np.nanmin(a), np.nanmax(a)
    den = mx - mn
    if (not np.isfinite(den)) or den <= 0:
        out = np.zeros_like(a)
        out[0] = 1e-6
        return out
    return (a - mn) / den


# =============================================================================
# I/O helpers
# =============================================================================

def load_v_linefit(path: str, nt: int, n_lines: int) -> np.ndarray:
    """
    Load cached LineFit LOS velocities from a .npy file.

    Accepts either shape (nt, n_lines) or (n_lines, nt) and returns (nt, n_lines).
    Raises ValueError for unexpected shapes.
    """
    v = np.asarray(np.load(path), float)
    if v.shape == (nt, n_lines):
        return v
    if v.shape == (n_lines, nt):
        return v.T
    raise ValueError(
        f"Unexpected v_linefit shape {v.shape}; "
        f"expected ({nt}, {n_lines}) or ({n_lines}, {nt})."
    )


def centers_from_velocity(lambda0_nm: np.ndarray, v_kms: np.ndarray) -> np.ndarray:
    """
    Convert LOS velocities (km/s) back to line-centre wavelengths (nm).

    Uses the non-relativistic Doppler formula:
        lambda_c = lambda0 * (1 + v / c)

    This is the exact inverse of the LineFit velocity convention:
        v = c * (lambda_c - lambda0) / lambda0
    so the round-trip lambda -> v -> lambda is lossless to floating-point precision.

    Parameters
    ----------
    lambda0_nm : (n_lines,) array
        Rest wavelengths in nm.
    v_kms : (nt, n_lines) array
        LOS velocities in km/s.

    Returns
    -------
    centers_nm : (nt, n_lines) array
    """
    return lambda0_nm[None, :] * (1.0 + v_kms / C_KMS)


# =============================================================================
# Coarse centre finder — close-pair-safe version
# =============================================================================

def coarse_centers_closepair_safe(
    wl: np.ndarray,
    I: np.ndarray,
    lambda0_nm: np.ndarray,
    search_half_win: int = 120,
    smooth: int = 3,
    close_pairs: Optional[List[Tuple[int, int]]] = None,
    peak_distance: int = 2,
    height_rel: float = 0.05,
    valley_pad_bins: int = 0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Localise absorption-core peaks in intensity-inverted space.

    Starting from expected rest wavelengths, this routine searches for peaks in
    inv = 1 - smooth(norm(I)), which correspond to absorption cores. For declared
    close pairs, ownership is enforced by splitting the shared search region at
    the inter-line valley, preventing centre swapping under moderate profile evolution.

    Parameters
    ----------
    wl : (nw,) array
        Wavelength axis in nm.
    I : (nw,) array
        Spectral intensity at one time step.
    lambda0_nm : (n_lines,) array
        Rest wavelengths used as prior centres.
    search_half_win : int
        Half-width of the search window around each prior centre (index bins).
    smooth : int
        Boxcar smoothing kernel size (bins) applied before inversion.
    close_pairs : list of (int, int), optional
        Pairs of line indices whose search regions are split at the valley.
    peak_distance : int
        Minimum separation between candidate peaks (bins).
    height_rel : float
        Minimum peak height relative to the local maximum.
    valley_pad_bins : int
        Padding added around the valley split point to prevent boundary effects.

    Returns
    -------
    centers_nm : (n_lines,) float array
        Coarse centre estimates in nm.
    idx : (n_lines,) int array
        Corresponding indices into the wavelength axis.
    """
    wl = np.asarray(wl, float)
    I  = np.asarray(I, float)
    lam0 = np.asarray(lambda0_nm, float)
    nlines = lam0.size
    n = wl.size

    if close_pairs is None:
        close_pairs = []

    # Replace non-finite values with median before normalisation
    y = np.nan_to_num(I, nan=np.nanmedian(I) if np.any(np.isfinite(I)) else 0.0)
    y01 = (y - np.min(y)) / (np.max(y) - np.min(y) + 1e-12)
    sm  = uniform_filter1d(y01, size=int(smooth))
    inv = 1.0 - sm   # peaks ↔ absorption cores

    def pick_peak_in_interval(i: int, lo: int, hi: int) -> int:
        """Return the best absorption-core peak index within [lo, hi]."""
        lo = int(max(0, lo))
        hi = int(min(n - 1, hi))
        if hi <= lo + 4:
            return int(np.clip(nearest_index(wl, lam0[i]), lo, hi))

        seg = inv[lo:hi + 1]
        if seg.size < 10 or not np.isfinite(seg).all():
            return int(lo + np.nanargmax(seg))

        thr  = float(height_rel * np.nanmax(seg))
        pks, _ = find_peaks(seg, distance=int(peak_distance), height=thr)
        if pks.size == 0:
            return int(lo + np.nanargmax(seg))

        pks_abs = lo + pks
        dist    = np.abs(wl[pks_abs] - lam0[i])
        height  = inv[pks_abs]
        # Score: prefer proximity to prior; minor preference for deep cores
        score = dist + 0.10 * (1.0 - height / (np.nanmax(height) + 1e-12))
        return int(pks_abs[np.argmin(score)])

    # --- Independent search for all lines ---
    idx = np.full(nlines, -1, dtype=int)
    for i in range(nlines):
        idx0 = nearest_index(wl, lam0[i])
        lo   = max(0, idx0 - search_half_win)
        hi   = min(n - 1, idx0 + search_half_win)
        idx[i] = pick_peak_in_interval(i, lo, hi)

    # --- Enforce ownership for declared close pairs ---
    for (ci, cj) in close_pairs:
        ii = nearest_index(wl, lam0[ci])
        jj = nearest_index(wl, lam0[cj])
        a, b = min(ii, jj), max(ii, jj)

        valley = (a + b) // 2 if (b <= a + 2) else int(a + np.argmin(inv[a:b + 1]))
        valley_L = max(0, valley - valley_pad_bins)
        valley_R = min(n - 1, valley + valley_pad_bins)

        lo_pair = max(0, a - search_half_win)
        hi_pair = min(n - 1, b + search_half_win)

        idx[ci] = pick_peak_in_interval(ci, lo_pair, valley_L)
        idx[cj] = pick_peak_in_interval(cj, valley_R, hi_pair)

    return wl[idx].astype(float), idx


# =============================================================================
# Morphology flags for the stress-case line
# =============================================================================

def morphology_flag_multipeak_scipy(
    window_inv01: np.ndarray,
    height_rel: float = 0.20,
) -> bool:
    """
    Return True if the intensity-inverted window contains two or more peaks
    exceeding a relative height threshold, indicating a multi-peak structure.
    """
    inv = np.asarray(window_inv01, float)
    if inv.size < 9 or not np.isfinite(inv).all():
        return False
    thr = float(height_rel) * float(np.nanmax(inv))
    if not (np.isfinite(thr) and thr > 0):
        return False
    pks, _ = find_peaks(inv, height=thr, distance=2)
    return bool(pks.size >= 2)


def morphology_flag_splitcore_strict(window_inv01: np.ndarray) -> bool:
    """
    Return True if the window shows a clear split-core structure:
    two prominent peaks whose inter-peak valley falls below a fraction of
    the weaker peak, indicating a genuine core bifurcation rather than noise.

    Uses module-level constants SPLIT_PROM_FRAC and SPLIT_VALLEY_FRAC.
    """
    inv = np.asarray(window_inv01, float)
    if inv.size < 9 or not np.isfinite(inv).all():
        return False

    dyn = float(np.nanmax(inv) - np.nanmin(inv))
    if not (np.isfinite(dyn) and dyn > 0):
        return False

    pks, props = find_peaks(inv, prominence=float(SPLIT_PROM_FRAC) * dyn, distance=2)
    if pks.size < 2:
        return False

    order = np.argsort(props["prominences"])[::-1]
    p1, p2 = sorted([int(pks[order[0]]), int(pks[order[1]])])

    valley = float(np.nanmin(inv[p1:p2 + 1]))
    pk1, pk2 = float(inv[p1]), float(inv[p2])

    if not (np.isfinite(valley) and np.isfinite(pk1) and np.isfinite(pk2)):
        return False

    return valley < float(SPLIT_VALLEY_FRAC) * min(pk1, pk2)


# =============================================================================
# Dataset construction
# =============================================================================

@dataclass
class BuiltData:
    """
    Container for the assembled ML dataset.

    Attributes
    ----------
    X : (N, 2, Lmax) float32
        Input tensor. Channel 0: intensity-inverted normalised window.
        Channel 1: validity mask (1.0 for true spectral samples, 0.0 for padding).
    y_pm : (N,) float32
        Supervised target: LineFit residual relative to coarse centre (pm).
    y_truth_pm : (N,) float32
        Ground-truth residual relative to coarse centre (pm; for verification only).
    time_idx : (N,) int16
        Time-step index for each sample (used for time-ordered splitting).
    line_idx : (N,) int16
        Line index for each sample (used for line-specific analysis).
    coarse_nm : (N,) float64
        Coarse centre wavelength (nm) for each sample.
    lam0_nm : (N,) float64
        Rest wavelength (nm) for each sample's line.
    """
    X:          np.ndarray
    y_pm:       np.ndarray
    y_truth_pm: np.ndarray
    time_idx:   np.ndarray
    line_idx:   np.ndarray
    coarse_nm:  np.ndarray
    lam0_nm:    np.ndarray


def build_dataset(
    spectra:           np.ndarray,
    wl_nm:             np.ndarray,
    lambda0_nm:        np.ndarray,
    centers_linefit_nm: np.ndarray,
    centers_truth_nm:  np.ndarray,
    per_line_halfwin:  List[int],
) -> BuiltData:
    """
    Construct the ML dataset from the spectral cube and pre-computed centres.

    For each (time step, line) pair:
      1. Compute a robust coarse centre from the current spectrum.
      2. Extract a local window of width (2*hw + 1) around the coarse centre.
      3. Intensity-invert and normalise the window to [0, 1].
      4. Pad to the maximum window size (Lmax = max over lines of 2*hw + 1)
         using edge-replication; build a validity mask to distinguish true
         spectral samples from padding regions.
      5. Compute the residual target = (LineFit centre − coarse centre) in pm.
      6. Compute the truth residual = (truth centre − coarse centre) in pm.

    The validity mask (channel 1) replaces the coordinate channel used in the
    original prototype. It explicitly encodes which positions in the padded
    window contain real spectral information, allowing the CNN to suppress
    padding artefacts without relying on implicit positional encoding.

    Parameters
    ----------
    spectra : (nt, nw) array
        Spectral cube.
    wl_nm : (nw,) array
        Wavelength axis in nm.
    lambda0_nm : (n_lines,) array
        Rest wavelengths in nm.
    centers_linefit_nm : (nt, n_lines) array
        LineFit line-centre estimates (training labels).
    centers_truth_nm : (nt, n_lines) or (n_lines, nt) array
        Ground-truth line centres (verification only).
    per_line_halfwin : list of int
        Per-line half-window sizes in index bins.

    Returns
    -------
    BuiltData
    """
    nt, nw = spectra.shape
    n_lines = int(lambda0_nm.size)

    # Ensure (nt, n_lines) layout for truth centres
    centers_truth_nm = np.asarray(centers_truth_nm, float)
    if centers_truth_nm.shape == (n_lines, nt):
        centers_truth_nm = centers_truth_nm.T
    elif centers_truth_nm.shape != (nt, n_lines):
        raise ValueError(
            f"Unexpected centers_truth_nm shape {centers_truth_nm.shape}; "
            f"expected ({nt}, {n_lines}) or ({n_lines}, {nt})."
        )

    # Maximum padded window length across all lines
    win_lens = np.array([2 * hw + 1 for hw in per_line_halfwin], dtype=int)
    Lmax = int(np.max(win_lens))

    N = nt * n_lines
    X        = np.zeros((N, 2, Lmax), dtype=np.float32)
    y_pm     = np.zeros(N, dtype=np.float32)
    y_truth  = np.zeros(N, dtype=np.float32)
    t_idx    = np.zeros(N, dtype=np.int16)
    l_idx    = np.zeros(N, dtype=np.int16)
    coarse   = np.zeros(N, dtype=np.float64)
    lam0_s   = np.zeros(N, dtype=np.float64)

    k = 0
    for t in range(nt):
        I_t = np.asarray(spectra[t], float)

        # Coarse centres for this time step
        coarse_nm_t, idx_ref = coarse_centers_closepair_safe(
            wl=wl_nm, I=I_t, lambda0_nm=lambda0_nm,
            search_half_win=SEARCH_HALF_WIN,
            smooth=SMOOTH_COARSE,
            close_pairs=CLOSE_PAIRS,
            peak_distance=2,
            height_rel=0.05,
            valley_pad_bins=0,
        )

        for i in range(n_lines):
            c0  = int(idx_ref[i])
            hw  = int(per_line_halfwin[i])
            L_i = 2 * hw + 1   # true window length for this line

            # Extract raw window (may be shorter near array edges)
            lo  = max(0, c0 - hw)
            hi  = min(nw, c0 + hw + 1)
            win = I_t[lo:hi].astype(np.float64)

            # Edge-replicate to fill L_i samples
            need = L_i - win.size
            if need > 0:
                padL = need // 2
                padR = need - padL
                win  = np.concatenate([
                    np.full(padL, win[0]  if win.size else 0.0),
                    win,
                    np.full(padR, win[-1] if win.size else 0.0),
                ])

            # --- Channel 0: inverted, normalised spectrum window ---
            win01  = minmax01(win)
            win_ch = (1.0 - win01).astype(np.float32)   # peaks = absorption cores

            # --- Channel 1: validity mask ---
            # 1.0 for positions that contain true spectral data, 0.0 for padding
            mask = np.zeros(L_i, dtype=np.float32)
            actual_lo = c0 - hw
            actual_hi = c0 + hw + 1   # exclusive
            # positions within L_i that map to valid spectrum indices
            valid_start = max(0,   hw - c0)            # offset into window
            valid_end   = min(L_i, nw - (c0 - hw))    # exclusive
            mask[valid_start:valid_end] = 1.0

            # Zero-pad both channels from L_i to Lmax (pad beyond true window)
            pad_to_Lmax = Lmax - L_i
            if pad_to_Lmax > 0:
                win_ch = np.concatenate([win_ch, np.zeros(pad_to_Lmax, dtype=np.float32)])
                mask   = np.concatenate([mask,   np.zeros(pad_to_Lmax, dtype=np.float32)])

            X[k, 0, :] = win_ch
            X[k, 1, :] = mask

            # Targets in picometres (residual relative to coarse centre)
            lam_coarse = float(coarse_nm_t[i])
            y_pm[k]    = float((centers_linefit_nm[t, i] - lam_coarse) * 1e3)
            y_truth[k] = float((centers_truth_nm[t, i]  - lam_coarse) * 1e3)

            t_idx[k]  = t
            l_idx[k]  = i
            coarse[k] = lam_coarse
            lam0_s[k] = float(lambda0_nm[i])
            k += 1

    return BuiltData(
        X=X, y_pm=y_pm, y_truth_pm=y_truth,
        time_idx=t_idx, line_idx=l_idx,
        coarse_nm=coarse, lam0_nm=lam0_s,
    )


def time_split_masks(
    data:                  BuiltData,
    train_frac:            float,
    cal_frac_within_train: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Construct Boolean masks for time-ordered train / calibration / validation splits.

    The split is strictly time-ordered to prevent information leakage from future
    time steps into the training or calibration sets. Within the training window
    the last cal_frac_within_train fraction is reserved for calibration (threshold
    selection) and receives no gradient updates.

    Returns
    -------
    m_train, m_cal, m_val : Boolean arrays of shape (N,)
    """
    nt = int(data.time_idx.max()) + 1
    t_cut  = int(np.floor(train_frac * nt))
    t_cal0 = int(np.floor((1.0 - cal_frac_within_train) * t_cut))

    m_train = data.time_idx < t_cal0
    m_cal   = (data.time_idx >= t_cal0) & (data.time_idx < t_cut)
    m_val   = data.time_idx >= t_cut

    return m_train, m_cal, m_val


# =============================================================================
# CNN model
# =============================================================================

class CNNLineEmbedWide(nn.Module):
    """
    Lightweight 1D CNN for line-centre residual prediction.

    Architecture overview
    ---------------------
    Input: (batch, 2, Lmax) — spectral window (channel 0) + validity mask (channel 1)
    Three convolutional blocks with BatchNorm and Dropout.
    Global average and max pooling concatenated ("avg+max" feature aggregation).
    Per-line embedding concatenated to pooled features.
    Skip connection from global-averaged raw input to the MLP head, improving
    gradient flow for the small network without adding significant parameters.
    MLP head → scalar residual shift (pm, in standardized units during training).

    Parameters
    ----------
    n_lines : int
        Number of spectral lines; sets the embedding table size.
    emb_dim : int
        Dimensionality of the per-line embedding vector.
    dropout_p : float
        Dropout probability used in all Dropout layers.
    """

    def __init__(self, n_lines: int, emb_dim: int = 16, dropout_p: float = 0.02):
        super().__init__()

        self.emb = nn.Embedding(n_lines, emb_dim)

        # --- Convolutional trunk ---
        self.conv1 = nn.Sequential(
            nn.Conv1d(2, 24, kernel_size=7, padding=3),
            nn.BatchNorm1d(24),
            nn.ReLU(),
            nn.Dropout(dropout_p),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(24, 48, kernel_size=5, padding=2),
            nn.BatchNorm1d(48),
            nn.ReLU(),
            nn.Dropout(dropout_p),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(48, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_p),
        )

        # Global pooling (avg + max concatenated)
        self.pool_avg = nn.AdaptiveAvgPool1d(1)
        self.pool_max = nn.AdaptiveMaxPool1d(1)

        # Skip connection: global average of raw input projected to skip_dim
        skip_dim = 16
        self.skip_proj = nn.Linear(2, skip_dim)   # applied after global avg of input

        # --- MLP head ---
        # Input size: 64*2 (avg+max) + emb_dim + skip_dim
        head_in = 64 * 2 + emb_dim + skip_dim
        self.head = nn.Sequential(
            nn.Linear(head_in, 96),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(96, 1),
        )

    def forward(self, x: torch.Tensor, line_idx: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (batch, 2, Lmax) tensor
        line_idx : (batch,) long tensor

        Returns
        -------
        pred : (batch,) tensor
            Predicted residual shift (standardized units).
        """
        # Skip: global average over spatial dimension → (batch, 2)
        skip_feat = x.mean(dim=2)                          # (batch, 2)
        skip_feat = self.skip_proj(skip_feat)              # (batch, skip_dim)

        # Convolutional trunk
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)

        # Global pooling
        a = self.pool_avg(h).squeeze(-1)                   # (batch, 64)
        m = self.pool_max(h).squeeze(-1)                   # (batch, 64)

        # Line embedding
        e = self.emb(line_idx)                             # (batch, emb_dim)

        # Concatenate all features including skip
        z = torch.cat([a, m, e, skip_feat], dim=1)        # (batch, head_in)
        return self.head(z).squeeze(-1)                    # (batch,)


# =============================================================================
# PyTorch Dataset wrapper
# =============================================================================

class NpDatasetTorch(Dataset):
    """Thin wrapper around NumPy arrays for use with torch DataLoader."""

    def __init__(self, X: np.ndarray, y: np.ndarray, line_idx: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.l = torch.from_numpy(line_idx).long()

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.l[idx], self.y[idx]


# =============================================================================
# Metric helpers
# =============================================================================

def metrics_pm(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Compute MAE, RMSE, and 95th-percentile absolute error (all in pm).

    Non-finite values are excluded from all calculations.
    """
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if np.count_nonzero(m) == 0:
        return {"mae_pm": np.nan, "rmse_pm": np.nan, "p95_abs_pm": np.nan}
    e = (y_pred[m] - y_true[m]).astype(np.float64)
    return {
        "mae_pm":      float(np.mean(np.abs(e))),
        "rmse_pm":     float(np.sqrt(np.mean(e * e))),
        "p95_abs_pm":  float(np.percentile(np.abs(e), 95)),
    }


def uncertainty_coverage(
    y_true:  np.ndarray,
    y_pred:  np.ndarray,
    std_est: np.ndarray,
) -> Dict[str, float]:
    """
    Compute empirical coverage of the MC-dropout uncertainty intervals.

    For a well-calibrated uncertainty estimate, the fraction of samples where
    |error| < k * std should approach the corresponding Gaussian coverage:
    68.3% at 1σ, 95.4% at 2σ, 99.7% at 3σ.

    Parameters
    ----------
    y_true, y_pred : arrays
        True and predicted values (pm).
    std_est : array
        Estimated standard deviation per sample (pm).

    Returns
    -------
    dict with keys "cov_1sigma", "cov_2sigma", "cov_3sigma" (fractions in [0, 1]).
    """
    m = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(std_est) & (std_est > 0)
    if np.count_nonzero(m) < 5:
        return {"cov_1sigma": np.nan, "cov_2sigma": np.nan, "cov_3sigma": np.nan}
    err = np.abs((y_pred[m] - y_true[m]).astype(np.float64))
    s   = std_est[m].astype(np.float64)
    return {
        "cov_1sigma": float(np.mean(err < 1.0 * s)),
        "cov_2sigma": float(np.mean(err < 2.0 * s)),
        "cov_3sigma": float(np.mean(err < 3.0 * s)),
    }


# =============================================================================
# MC-dropout inference (BatchNorm-safe)
# =============================================================================

def _enable_dropout_only(model: nn.Module) -> None:
    """
    Set model to eval mode (freezing BatchNorm running statistics), then
    re-enable all Dropout layers for stochastic forward passes.

    The procedure for MC-dropout: BatchNorm uses its
    accumulated running statistics (eval mode), while Dropout remains
    active to provide the stochastic ensemble. 
    """
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


@torch.no_grad()
def mc_dropout_predict(
    model:    nn.Module,
    X:        np.ndarray,
    line_idx: np.ndarray,
    passes:   int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Estimate predictive mean and standard deviation via MC-dropout.

    BatchNorm layers are kept in eval mode throughout; only Dropout is active.

    Parameters
    ----------
    model : nn.Module
        Trained model with Dropout layers.
    X : (N, 2, Lmax) float32 array
    line_idx : (N,) int64 array
    passes : int
        Number of stochastic forward passes.

    Returns
    -------
    mean_pred : (N,) float32 array
    std_pred  : (N,) float32 array
    """
    _enable_dropout_only(model)

    Xt = torch.from_numpy(X).float()
    Lt = torch.from_numpy(line_idx).long()

    preds = np.stack(
        [model(Xt, Lt).cpu().numpy().astype(np.float32) for _ in range(int(passes))],
        axis=0,
    )   # (passes, N)

    # Restore eval mode after passes
    model.eval()
    return np.mean(preds, axis=0), np.std(preds, axis=0)


# =============================================================================
# EMA (Exponential Moving Average) of model weights
# =============================================================================

class EMA:
    """
    Maintains an EMA shadow copy of model parameters.

    The EMA weights typically produce smoother loss landscapes and slightly
    better generalisation than the best-checkpoint weights, particularly with
    small batch sizes and ReduceLROnPlateau schedulers.
    """

    def __init__(self, model: nn.Module, decay: float = 0.995):
        self.decay  = float(decay)
        self.shadow: Dict[str, torch.Tensor] = {
            name: p.detach().clone()
            for name, p in model.named_parameters()
            if p.requires_grad
        }

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        """Update shadow weights: shadow = decay * shadow + (1 - decay) * param."""
        d = self.decay
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.shadow[name].mul_(d).add_(p.detach(), alpha=(1.0 - d))

    def copy_to(self, model: nn.Module) -> None:
        """Copy shadow weights into model parameters (in-place)."""
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad and name in self.shadow:
                    p.copy_(self.shadow[name])


# =============================================================================
# Standardization helpers
# =============================================================================

def build_standardization(
    y_train:      np.ndarray,
    L_train:      np.ndarray,
    n_lines:      int,
    sig_floor_pm: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute per-line mean and standard deviation from training targets.

    The standard deviation is floored at sig_floor_pm to prevent instability
    when a line has very low target variance (e.g. isolated lines with highly
    consistent LineFit outputs).

    Returns
    -------
    mu  : (n_lines,) float64 — per-line mean
    sig : (n_lines,) float64 — per-line std (floored at sig_floor_pm)
    """
    mu  = np.zeros(n_lines, float)
    sig = np.ones(n_lines, float)
    for i in range(n_lines):
        mli = (L_train == i)
        if np.count_nonzero(mli) > 5:
            mu[i]  = float(np.mean(y_train[mli]))
            s      = float(np.std(y_train[mli]))
            sig[i] = max(s if s > 1e-6 else 1.0, float(sig_floor_pm))
    return mu, sig


def standardize(y: np.ndarray, L: np.ndarray, mu: np.ndarray, sig: np.ndarray) -> np.ndarray:
    return ((y - mu[L]) / sig[L]).astype(np.float32)


def unstandardize(y_z: np.ndarray, L: np.ndarray, mu: np.ndarray, sig: np.ndarray) -> np.ndarray:
    return (y_z * sig[L] + mu[L]).astype(np.float32)


# =============================================================================
# Deterministic batch inference
# =============================================================================

@torch.no_grad()
def eval_model(
    model:        nn.Module,
    device:       torch.device,
    X:            np.ndarray,
    L:            np.ndarray,
    mu:           np.ndarray,
    sig:          np.ndarray,
    standardized: bool,
) -> np.ndarray:
    """
    Run deterministic (eval-mode) inference and return predictions in pm.

    Parameters
    ----------
    model : nn.Module (in eval mode on return)
    device : torch.device
    X : (N, 2, Lmax) float32 array
    L : (N,) int64 array — line indices
    mu, sig : (n_lines,) arrays — standardization parameters
    standardized : bool — whether the model was trained on standardized targets

    Returns
    -------
    preds_pm : (N,) float32 array
    """
    model.eval()
    Xt = torch.from_numpy(X).float().to(device)
    Lt = torch.from_numpy(L).long().to(device)
    y_z = model(Xt, Lt).cpu().numpy().astype(np.float32)
    return unstandardize(y_z, L, mu, sig) if standardized else y_z


# =============================================================================
# Temporal smoothness penalty helpers
# =============================================================================

def _sorted_stress_indices(
    m_train: np.ndarray,
    data:    BuiltData,
    line_i:  int,
) -> np.ndarray:
    """Return training sample indices for the stress line, sorted by time."""
    idx = np.where(m_train & (data.line_idx == int(line_i)))[0]
    return idx[np.argsort(data.time_idx[idx].astype(np.int64))] if idx.size else idx


def _stress_diff_loss(
    model:        nn.Module,
    device:       torch.device,
    data:         BuiltData,
    idx_sorted:   np.ndarray,
    mu:           np.ndarray,
    sig:          np.ndarray,
    standardized: bool,
) -> torch.Tensor:
    """
    Temporal smoothness regularizer for the stress-case line.

    Computes the mean squared first difference of the model's prediction error
    on the stress-line training samples (ordered by time). This penalises
    solutions whose errors oscillate rapidly in time, encouraging the emulator
    to produce temporally coherent outputs even for complex profiles.

    This penalty is evaluated and accumulated within the same backward pass as
    the main Huber loss (via the caller), rather than in a separate optimizer
    step, to avoid conflicting gradient updates.
    """
    if idx_sorted.size < 3:
        return torch.tensor(0.0, device=device)

    Xs = data.X[idx_sorted].astype(np.float32)
    Ls = data.line_idx[idx_sorted].astype(np.int64)
    yt = data.y_truth_pm[idx_sorted].astype(np.float32)

    if standardized:
        yt = ((yt - mu[Ls]) / sig[Ls]).astype(np.float32)

    Xt = torch.from_numpy(Xs).float().to(device)
    Lt = torch.from_numpy(Ls).long().to(device)
    yt_t = torch.from_numpy(yt).float().to(device)

    pred = model(Xt, Lt)
    err  = pred - yt_t
    de   = err[1:] - err[:-1]
    return torch.mean(de * de)


# =============================================================================
# RGWS (Refined Global Wavelet Spectrum) helper
# =============================================================================

def compute_rgws(
    times_s: np.ndarray,
    v_kms:   np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute the RGWS of a velocity time series using WaLSAtools.

    Requires WaLSAtools to be installed. Raises RuntimeError otherwise.

    Parameters
    ----------
    times_s : (nt,) array — time axis in seconds
    v_kms   : (nt,) array — LOS velocity in km/s

    Returns
    -------
    f_mhz : frequency axis in mHz
    power : RGWS power (arbitrary units)
    """
    if not HAVE_WALSA:
        raise RuntimeError("WaLSAtools is not installed; cannot compute RGWS.")
    v_apod = WaLSA_detrend_apod(v_kms, apod=APOD, silent=True)
    out = WaLSAtools(
        signal=v_apod, time=times_s, method="wavelet",
        siglevel=SIGLEVEL, apod=APOD, mother=MOTHER,
        GWS=True, RGWS=True,
        cadence=float(np.nanmedian(np.diff(times_s))),
        silent=True, nodetrendapod=True,
    )
    periods_s = np.asarray(out[1], float)
    power     = np.asarray(out[-1], float)
    return 1e3 / periods_s, power   # convert periods to mHz


# =============================================================================
# Main training function
# =============================================================================

def train_model(
    data:    BuiltData,
    n_lines: int,
    config:  Dict[str, Any],
    epochs:  int,
    patience: int,
    seed:    int,
) -> Dict[str, Any]:
    """
    Train the CNN emulator with the fixed paper configuration.

    Training procedure
    ------------------
    1. Build time-ordered splits (train / cal / val).
    2. Standardize targets per line using training-set statistics only.
    3. Train with AdamW + ReduceLROnPlateau + gradient clipping.
    4. Apply EMA weight averaging after each optimizer step.
    5. The temporal smoothness penalty for the stress line is accumulated
       into the *same* backward pass as the Huber loss (one optimizer step),
       avoiding conflicting gradients from separate backward passes.
    6. Early stopping on validation Huber loss with patience.
    7. Save best checkpoint and best EMA checkpoint independently.
    8. Evaluate both against the validation set vs ground truth.

    Returns
    -------
    dict containing model state, standardization parameters, metrics,
    and per-epoch training curves.
    """
    set_seed(seed)
    if TORCH_NUM_THREADS is not None:
        torch.set_num_threads(int(TORCH_NUM_THREADS))

    device     = torch.device("cpu")
    stress_lam = float(config.get("stress_diff_lambda", 0.0))

    m_train, m_cal, m_val = time_split_masks(
        data,
        train_frac=TRAIN_FRAC_TIME,
        cal_frac_within_train=CAL_FRAC_WITHIN_TRAIN,
    )

    Xtr  = data.X[m_train];      Ltr  = data.line_idx[m_train].astype(np.int64)
    Xca  = data.X[m_cal];        Lca  = data.line_idx[m_cal].astype(np.int64)
    Xva  = data.X[m_val];        Lva  = data.line_idx[m_val].astype(np.int64)
    ytr  = data.y_pm[m_train];   yca  = data.y_pm[m_cal];   yva  = data.y_pm[m_val]
    ytr_truth = data.y_truth_pm[m_train]
    yca_truth = data.y_truth_pm[m_cal]
    yva_truth = data.y_truth_pm[m_val]

    # --- Per-line standardization (training set only) ---
    if STANDARDIZE_TARGET_PER_LINE:
        mu, sig    = build_standardization(ytr, Ltr, n_lines, float(config["sig_floor_pm"]))
        ytr_z      = standardize(ytr, Ltr, mu, sig)
        yca_z      = standardize(yca, Lca, mu, sig)
        yva_z      = standardize(yva, Lva, mu, sig)
        standardized = True
    else:
        mu, sig      = np.zeros(n_lines, float), np.ones(n_lines, float)
        ytr_z, yca_z, yva_z = ytr.astype(np.float32), yca.astype(np.float32), yva.astype(np.float32)
        standardized = False

    # Sorted stress-line training indices (needed for temporal penalty)
    idx_stress_sorted = _sorted_stress_indices(m_train, data, LINE_STRESS)

    # --- Model, optimiser, scheduler ---
    model = CNNLineEmbedWide(
        n_lines=n_lines,
        emb_dim=int(config["emb_dim"]),
        dropout_p=float(config["dropout_p"]),
    ).to(device)

    ema       = EMA(model, decay=float(config.get("ema_decay", 0.995))) if USE_EMA else None
    loss_fn   = nn.SmoothL1Loss(beta=float(config["huber_beta"]))
    opt       = torch.optim.AdamW(
        model.parameters(),
        lr=float(config["lr"]),
        weight_decay=float(config["weight_decay"]),
        eps=1e-8,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=4, min_lr=1e-5,
    )

    train_loader = DataLoader(
        NpDatasetTorch(Xtr, ytr_z, Ltr),
        batch_size=int(config["batch_size"]),
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        NpDatasetTorch(Xva, yva_z, Lva),
        batch_size=int(config["batch_size"]),
        shuffle=False,
        drop_last=False,
    )

    best_val     = np.inf
    best_state   = None
    best_ema_st  = None
    bad          = 0
    IMPROVE_EPS  = 1e-5

    # Training curve accumulators
    train_losses_ep: List[float] = []
    val_losses_ep:   List[float] = []

    # Pre-compute stress-line tensors once (they are fixed per epoch)
    if (stress_lam > 0.0) and (idx_stress_sorted.size >= 3):
        Xs_t  = torch.from_numpy(data.X[idx_stress_sorted].astype(np.float32)).to(device)
        Ls_t  = torch.from_numpy(data.line_idx[idx_stress_sorted].astype(np.int64)).to(device)
        yt_np = data.y_truth_pm[idx_stress_sorted].astype(np.float32)
        if standardized:
            yt_np = ((yt_np - mu[data.line_idx[idx_stress_sorted]]) /
                     sig[data.line_idx[idx_stress_sorted]]).astype(np.float32)
        yt_t = torch.from_numpy(yt_np).to(device)
        use_stress_penalty = True
    else:
        use_stress_penalty = False

    for ep in range(1, int(epochs) + 1):
        model.train()
        ep_train_losses: List[float] = []

        for xb, lb, yb in train_loader:
            xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)

            opt.zero_grad(set_to_none=True)

            pred  = model(xb, lb)
            loss  = loss_fn(pred, yb)

            # --- Temporal smoothness penalty: accumulated in the SAME backward pass ---
            # This avoids conflicting gradients from a separate optimizer step.
            if use_stress_penalty:
                model_pred_stress = model(Xs_t, Ls_t)
                err_s  = model_pred_stress - yt_t
                de     = err_s[1:] - err_s[:-1]
                loss   = loss + stress_lam * torch.mean(de * de)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            if ema is not None:
                ema.update(model)

            ep_train_losses.append(float(loss.detach().cpu().numpy()))

        train_losses_ep.append(float(np.mean(ep_train_losses)))

        # --- Validation loss ---
        model.eval()
        va_losses: List[float] = []
        with torch.no_grad():
            for xb, lb, yb in val_loader:
                xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
                va_losses.append(float(loss_fn(model(xb, lb), yb).cpu().numpy()))
        va_loss = float(np.mean(va_losses)) if va_losses else np.nan
        val_losses_ep.append(va_loss)

        scheduler.step(va_loss)

        # --- Checkpoint ---
        if np.isfinite(va_loss) and va_loss < (best_val - IMPROVE_EPS):
            best_val   = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            if ema is not None:
                tmp = CNNLineEmbedWide(
                    n_lines=n_lines, emb_dim=int(config["emb_dim"]),
                    dropout_p=float(config["dropout_p"]),
                )
                tmp.load_state_dict(best_state)
                ema.copy_to(tmp)
                best_ema_st = {k: v.detach().cpu().clone() for k, v in tmp.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if bad >= int(patience):
            print(f"  Early stopping at epoch {ep} (patience={patience}).")
            break

    assert best_state is not None, "Training produced no valid checkpoint."

    # --- Evaluate best vs EMA on validation set against truth ---
    packs: Dict[str, Dict] = {"best": best_state}
    if best_ema_st is not None:
        packs["ema"] = best_ema_st

    evals_vs_truth: Dict[str, Dict] = {}
    for tag, st in packs.items():
        m = CNNLineEmbedWide(n_lines, int(config["emb_dim"]), float(config["dropout_p"])).to(device)
        m.load_state_dict(st)
        yh = eval_model(m, device, Xva, Lva, mu, sig, standardized)
        evals_vs_truth[tag] = metrics_pm(yva_truth, yh)

    # --- Pick final weights ---
    picked_tag   = PICK_WEIGHTS_TAG if PICK_WEIGHTS_TAG in packs else "best"
    picked_state = packs[picked_tag]

    # --- Final metrics with picked weights ---
    mfinal = CNNLineEmbedWide(n_lines, int(config["emb_dim"]), float(config["dropout_p"])).to(device)
    mfinal.load_state_dict(picked_state)

    yhat_va       = eval_model(mfinal, device, Xva, Lva, mu, sig, standardized)
    yhat_ca       = eval_model(mfinal, device, Xca, Lca, mu, sig, standardized)
    val_vs_linefit = metrics_pm(yva, yhat_va)
    val_vs_truth   = metrics_pm(yva_truth, yhat_va)
    cal_vs_truth   = metrics_pm(yca_truth, yhat_ca)

    return {
        "config":       dict(config),
        "seed":         int(seed),
        "picked_tag":   picked_tag,
        "picked_state": picked_state,
        "mu":           mu,
        "sig":          sig,
        "standardized": bool(standardized),
        "val_vs_linefit": val_vs_linefit,
        "val_vs_truth":   val_vs_truth,
        "cal_vs_truth":   cal_vs_truth,
        "val_vs_truth_allpacks": evals_vs_truth,
        "best_val_loss": float(best_val),
        "train_losses":  train_losses_ep,
        "val_losses":    val_losses_ep,
    }


# =============================================================================
# Plotting helpers
# =============================================================================

def _savefig(path: str) -> None:
    """Save current matplotlib figure to path and close."""
    if HAVE_MPL:
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()


def plot_val_hist_cdf(
    outdir:             str,
    e_abs_emul_truth:   np.ndarray,
    e_abs_emul_linefit: np.ndarray,
    e_abs_hyb_truth:    np.ndarray,
) -> None:
    """Plot histogram and CDF of absolute centre errors on the validation set."""
    if not HAVE_MPL:
        return

    def cdf(x):
        x = np.sort(x[np.isfinite(x)])
        return x, np.linspace(0.0, 1.0, x.size)

    plt.figure(figsize=(11, 5))

    plt.subplot(1, 2, 1)
    plt.hist(e_abs_emul_truth,   bins=60, alpha=0.6, label="|Emul − Truth|")
    plt.hist(e_abs_hyb_truth,    bins=60, alpha=0.6, label="|Hybrid − Truth|")
    plt.hist(e_abs_emul_linefit, bins=60, alpha=0.35, label="|Emul − LineFit|")
    plt.xlabel("Absolute error (pm)")
    plt.ylabel("Count")
    plt.title("Validation absolute error histograms")
    plt.legend(frameon=False)

    plt.subplot(1, 2, 2)
    x1, y1 = cdf(e_abs_emul_truth)
    x2, y2 = cdf(e_abs_hyb_truth)
    x3, y3 = cdf(e_abs_emul_linefit)
    plt.plot(x1, y1, "-",   label="|Emul − Truth| CDF")
    plt.plot(x2, y2, "--",  label="|Hybrid − Truth| CDF")
    plt.plot(x3, y3, "-.",  label="|Emul − LineFit| CDF")
    plt.xlabel("Absolute error (pm)")
    plt.ylabel("CDF")
    plt.title("Validation CDFs")
    plt.legend(frameon=False)

    _savefig(os.path.join(outdir, "val_error_hist_cdf.pdf"))


def plot_fallback_map(
    outdir:   str,
    data:     BuiltData,
    fallback: np.ndarray,
) -> None:
    """Scatter plot of fallback sample positions in (time index, line index) space."""
    if not HAVE_MPL:
        return
    idx = np.where(fallback)[0]
    plt.figure(figsize=(10, 4.5))
    if idx.size > 0:
        plt.plot(data.time_idx[idx].astype(int), data.line_idx[idx].astype(int),
                 ".", markersize=3.0)
    plt.xlabel("Time index")
    plt.ylabel("Line index")
    plt.title(f"Fallback map — {idx.size} samples ({100.0 * idx.size / data.X.shape[0]:.1f}%)")
    _savefig(os.path.join(outdir, "fallback_map.pdf"))


def _interp_to_ref(f_src, P_src, f_ref):
    """Interpolate (f_src, P_src) onto f_ref; returns NaN outside input range."""
    m = np.isfinite(f_src) & np.isfinite(P_src)
    if np.count_nonzero(m) < 2:
        return np.full_like(f_ref, np.nan, float)
    o = np.argsort(f_src[m])
    return np.interp(f_ref, f_src[m][o], P_src[m][o], left=np.nan, right=np.nan)


def plot_rgws_line6(
    outdir:    str,
    time_s:    np.ndarray,
    v_truth:   np.ndarray,
    v_linefit: np.ndarray,
    v_emul:    np.ndarray,
    v_hyb:     np.ndarray,
) -> None:
    """RGWS comparison for the stress-case line: Truth, LineFit, Emulator, Hybrid."""
    if not (HAVE_MPL and HAVE_WALSA):
        return

    fT, PT = compute_rgws(time_s, v_truth)
    fL, PL = compute_rgws(time_s, v_linefit)
    fE, PE = compute_rgws(time_s, v_emul)
    fH, PH = compute_rgws(time_s, v_hyb)

    PLi = _interp_to_ref(fL, PL, fT)
    PEi = _interp_to_ref(fE, PE, fT)
    PHi = _interp_to_ref(fH, PH, fT)

    plt.figure(figsize=(10, 5))
    plt.plot(fT, PT,  "-",  label="Truth")
    plt.plot(fT, PLi, "--", label="LineFit")
    plt.plot(fT, PEi, "-.", label="CNN emulator")
    plt.plot(fT, PHi, ":",  label="Hybrid")
    plt.xlim(0.1, 15.0)
    plt.xlabel("Frequency (mHz)")
    plt.ylabel("RGWS power (arb.)")
    plt.title("PoP: RGWS preservation — stress-case line")
    plt.legend(frameon=False)
    _savefig(os.path.join(outdir, "rgws_line6_comparison.pdf"))


# =============================================================================
# LineFit fallback runner (for directly measured hybrid timing)
# =============================================================================

def _run_linefit_on_timesteps(
    timestep_indices: np.ndarray,
    spectra:          np.ndarray,
    wl_nm:            np.ndarray,
    lambda0_nm:       np.ndarray,
    line_index:       int,
) -> Tuple[np.ndarray, float]:
    """
    Run LineFit on a specified subset of time steps, here for ONE line only,
    and measure the wall-clock time.

    Since the hybrid fallback in this PoP applies only to the stress-case line
    (LINE_STRESS), there is no reason to fit all 10 lines for every fallback
    time step. Only the stress-case line's rest wavelength, window parameters,
    and reversal configuration are passed, reducing the per-step cost by ~10×
    compared to a full 10-line fit and giving a timing figure that accurately
    reflects what the production hybrid would actually invoke.

    All console output is suppressed via stdout redirection in addition to
    silent=True, to prevent reversal-detector debug prints from appearing
    during the timing run.

    Parameters
    ----------
    timestep_indices : (k,) int array
        Indices into the time axis of spectra to process.
    spectra : (nt, nw) array
    wl_nm   : (nw,) array
    lambda0_nm : (n_lines,) array — full rest-wavelength array; only
        lambda0_nm[line_index] is passed to LineFit.
    line_index : int
        0-based index of the line to fit (must match LINE_STRESS).

    Returns
    -------
    v_out     : float array — LOS velocities for processed steps (1 line)
    elapsed_s : float       — wall-clock time in seconds
    """
    import io
    import contextlib

    # Extract single-line parameters — only the stress-case line is fitted
    lam0_single   = np.array([lambda0_nm[line_index]], dtype=np.float64)
    element_single = np.array([f"Line {line_index}"], dtype=object)

    # Per-line arrays sliced to the single stress-case line
    win_single    = [_LF_PER_LINE_WIN[line_index]]
    min_hw_single = [_LF_PER_LINE_MIN_HW[line_index]]
    max_hw_single = [_LF_PER_LINE_MAX_HW[line_index]]

    # reversal_lines: if the stress line is a reversal candidate, keep it;
    # re-index to 0 since it is now the only line passed to LineFit
    reversal_single = [0] if (line_index in _LF_REVERSAL_LINES) else []

    # close_pairs: drop any pair involving the stress line (it has no partner
    # in the single-line call) — none of the declared pairs involve LINE_STRESS
    # in the current testbed, so this list will be empty
    close_pairs_single = []

    _sink = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(_sink):
        v_out, _ = linefit(
            spectra               = np.asarray(spectra, dtype=np.float64),
            fit_func              = "asymmetric_voigt",
            num_iterations        = _LF_NUM_ITERATIONS,
            date_identifier       = "hybrid_fallback_timing",
            refwavelength         = lam0_single,
            wavelengths           = np.asarray(wl_nm, dtype=np.float64),
            element               = element_single,
            pixel                 = np.asarray(timestep_indices, dtype=int),
            window_size           = _LF_WINDOW_SIZE,
            per_line_window_size  = win_single,
            per_line_min_halfwin  = min_hw_single,
            per_line_max_halfwin  = max_hw_single,
            smoothing_window      = _LF_SMOOTHING_WINDOW,
            return_vlos_intensity = True,
            adaptive_window       = _LF_ADAPTIVE_WIN,
            use_coarse_centers    = _LF_USE_COARSE,
            close_pairs           = close_pairs_single,
            reversal_lines        = reversal_single,
            write_csv             = _LF_WRITE_CSV,
            save_fitted_spectra   = _LF_SAVE_FITTED,
            silent                = _LF_SILENT,
        )
    elapsed_s = float(time.perf_counter() - t0)
    return np.asarray(v_out, float), elapsed_s


# =============================================================================
# Main entry point
# =============================================================================

def main() -> None:
    """
    Execute the full proof-of-principle emulator pipeline:
      1. Load FITS testbed and cached LineFit outputs.
      2. Build ML dataset (residual-to-coarse formulation).
      3. Train fixed-configuration CNN emulator.
      4. Benchmark emulator inference speed.
      5. Apply hybrid gating (morphology + MC-dropout) to stress-case line.
      6. Compute and save all metrics, diagnostics, and plots.
    """
    os.makedirs(OUTDIR,  exist_ok=True)
    os.makedirs(PLOTDIR, exist_ok=True)

    if TORCH_NUM_THREADS is not None:
        torch.set_num_threads(int(TORCH_NUM_THREADS))

    # -------------------------------------------------------------------------
    # 1. Load data
    # -------------------------------------------------------------------------
    print("Loading FITS testbed...")
    with fits.open(FITS_PATH) as hdul:
        spectra        = np.asarray(hdul[0].data,          dtype=np.float64)
        wl_nm          = np.asarray(hdul["WAVELENGTH"].data, dtype=np.float64)
        time_s         = np.asarray(hdul["TIME"].data,       dtype=np.float64)
        lambda0_nm     = np.asarray(hdul["LAMBDA0"].data,    dtype=np.float64)
        center_true_nm = np.asarray(hdul["CENTER_TRUE"].data, dtype=np.float64)

    nt, _nw = spectra.shape
    n_lines = int(lambda0_nm.size)
    print(f"  Loaded: {nt} time steps, {_nw} wavelength channels, {n_lines} lines.")

    if not os.path.exists(V_LINEFIT_NPY):
        raise FileNotFoundError(
            f"Cached LineFit file not found: {V_LINEFIT_NPY}\n"
            "Run the LineFit analysis and cache the velocity array before proceeding."
        )

    print("Loading cached LineFit velocities...")
    v_linefit         = load_v_linefit(V_LINEFIT_NPY, nt=nt, n_lines=n_lines)
    centers_linefit_nm = centers_from_velocity(lambda0_nm, v_linefit)

    # -------------------------------------------------------------------------
    # 2. Build ML dataset
    # -------------------------------------------------------------------------
    print("Building ML dataset (residual-to-coarse formulation)...")
    data = build_dataset(
        spectra=spectra,
        wl_nm=wl_nm,
        lambda0_nm=lambda0_nm,
        centers_linefit_nm=centers_linefit_nm,
        centers_truth_nm=center_true_nm,
        per_line_halfwin=PER_LINE_HALF_WIN,
    )
    print(f"  Dataset: {data.X.shape[0]} samples, input shape {data.X.shape[1:]}")

    # -------------------------------------------------------------------------
    # 3. Train emulator
    # -------------------------------------------------------------------------
    print(f"\nTraining CNN emulator (seed={FINAL_SEED}, epochs={FINAL_EPOCHS})...")
    result = train_model(
        data=data, n_lines=n_lines,
        config=FINAL_CFG,
        epochs=FINAL_EPOCHS,
        patience=FINAL_PATIENCE,
        seed=FINAL_SEED,
    )

    picked_tag   = result["picked_tag"]
    model_state  = result["picked_state"]
    mu           = result["mu"]
    sig          = result["sig"]
    standardized = result["standardized"]

    print(f"  Training complete. Picked weights: '{picked_tag}'")
    print(f"  VAL vs LineFit: {result['val_vs_linefit']}")
    print(f"  VAL vs Truth:   {result['val_vs_truth']}")

    # Save model
    torch.save(
        {
            "state_dict":  model_state,
            "mu":          mu,
            "sig":         sig,
            "standardize": standardized,
            "config":      FINAL_CFG,
            "seed":        FINAL_SEED,
            "picked_tag":  picked_tag,
            "n_lines":     n_lines,
            "Lmax":        int(data.X.shape[2]),
        },
        os.path.join(OUTDIR, "cnn_model.pt"),
    )

    # Save training curves (for convergence audit — data saved, no plot needed)
    np.savez(
        os.path.join(OUTDIR, "training_curves.npz"),
        train_losses=np.array(result["train_losses"], float),
        val_losses=np.array(result["val_losses"],   float),
    )

    # -------------------------------------------------------------------------
    # 4. Rebuild validation arrays for diagnostics
    # -------------------------------------------------------------------------
    m_train, m_cal, m_val = time_split_masks(
        data,
        train_frac=TRAIN_FRAC_TIME,
        cal_frac_within_train=CAL_FRAC_WITHIN_TRAIN,
    )
    Xva         = data.X[m_val]
    Lva         = data.line_idx[m_val].astype(np.int64)
    yva_linefit = data.y_pm[m_val]
    yva_truth   = data.y_truth_pm[m_val]

    device = torch.device("cpu")
    model  = CNNLineEmbedWide(
        n_lines=n_lines,
        emb_dim=int(FINAL_CFG["emb_dim"]),
        dropout_p=float(FINAL_CFG["dropout_p"]),
    ).to(device)
    model.load_state_dict(model_state)
    model.eval()

    yhat_va       = eval_model(model, device, Xva, Lva, mu, sig, standardized)
    m_vs_linefit  = metrics_pm(yva_linefit, yhat_va)
    m_vs_truth    = metrics_pm(yva_truth,   yhat_va)

    # -------------------------------------------------------------------------
    # 5. Emulator inference benchmark
    #    The fallback LineFit timing is measured AFTER Step 6 once the actual
    #    fallback time steps are known (see Step 5b below).
    # -------------------------------------------------------------------------
    print("\nBenchmarking emulator inference...")
    Xt_va = torch.from_numpy(Xva).float().to(device)
    Lt_va = torch.from_numpy(Lva).long().to(device)

    traced, use_traced = None, False
    if USE_TORCHSCRIPT_TRACE_FOR_TIMING:
        try:
            traced     = torch.jit.trace(model, (Xt_va, Lt_va)).eval()
            use_traced = True
        except Exception as exc:
            warnings.warn(f"TorchScript trace failed ({exc}); using eager mode for timing.")

    runner = traced if (use_traced and traced is not None) else model

    # Warm up
    with torch.inference_mode():
        for _ in range(TIMING_WARMUP):
            runner(Xt_va, Lt_va)
    # Val-set benchmark (for per-sample µs figure reported in the paper)
    t0_val = time.perf_counter()
    with torch.inference_mode():
        for _ in range(INFER_REPEATS):
            runner(Xt_va, Lt_va)
    infer_total_s = time.perf_counter() - t0_val

    infer_per_pass_ms   = 1e3 * infer_total_s / max(INFER_REPEATS, 1)
    infer_per_sample_us = (infer_per_pass_ms * 1e3) / max(int(Xva.shape[0]), 1)
    print(f"  Emulator (val-set, {int(Xva.shape[0])} samples): "
          f"{infer_per_pass_ms:.2f} ms/pass, {infer_per_sample_us:.2f} µs/sample")

    # Full-dataset inference time (all 3000 samples — the emulator component
    # of the hybrid in production)
    X_all_t = torch.from_numpy(data.X).float().to(device)
    L_all_t = torch.from_numpy(data.line_idx.astype(np.int64)).long().to(device)
    with torch.inference_mode():
        for _ in range(5):
            runner(X_all_t, L_all_t)
    t0_full = time.perf_counter()
    with torch.inference_mode():
        for _ in range(20):
            runner(X_all_t, L_all_t)
    t_full_emul_s = (time.perf_counter() - t0_full) / 20.0
    print(f"  Emulator (full dataset, {data.X.shape[0]} samples): "
          f"{1e3*t_full_emul_s:.2f} ms per pass")

    # -------------------------------------------------------------------------
    # 6. Hybrid gating (stress-case line only)
    # -------------------------------------------------------------------------
    print("\nApplying hybrid gating to stress-case line...")

    # --- Morphology flags (all samples, stress-line only) ---
    morph_flag = np.zeros(data.X.shape[0], dtype=bool)
    idx_stress = np.where(data.line_idx == int(LINE_STRESS))[0]
    for k in idx_stress:
        w0 = data.X[k, 0, :]   # channel 0: inverted spectrum
        morph_flag[k] = (
            morphology_flag_splitcore_strict(w0) or
            morphology_flag_multipeak_scipy(w0, height_rel=float(MULTIPEAK_HEIGHT_REL))
        )

    # --- Calibrate uncertainty threshold on CAL ∩ stress-line ∩ morph_flag ---
    m_cal_morph = m_cal & morph_flag
    n_cal_morph = int(np.count_nonzero(m_cal_morph))

    if n_cal_morph >= 5:
        _, std_cal_z = mc_dropout_predict(
            model,
            data.X[m_cal_morph],
            data.line_idx[m_cal_morph].astype(np.int64),
            passes=MC_DROPOUT_PASSES,
        )
        std_cal = (std_cal_z * sig[data.line_idx[m_cal_morph]]).astype(np.float32) \
            if standardized else std_cal_z.astype(np.float32)
        UNC_PM_THRESH = float(np.nanpercentile(std_cal, 95))
        unc_thresh_rule = f"P95 of MC-dropout std on {n_cal_morph} morph-flagged CAL samples"
    else:
        # Guard: if no CAL morph samples, disable fallback rather than using
        # threshold=0 which would trigger on all morphology-flagged samples.
        UNC_PM_THRESH  = np.inf
        unc_thresh_rule = (
            f"Fallback disabled: only {n_cal_morph} morph-flagged CAL samples "
            "(need >= 5 to calibrate threshold)."
        )
        warnings.warn(unc_thresh_rule)

    print(f"  UNC_PM_THRESH = {UNC_PM_THRESH:.4f} pm  ({unc_thresh_rule})")

    # --- MC-dropout on all morphology-flagged samples ---
    std_all = np.full(data.X.shape[0], np.nan, dtype=np.float32)
    if morph_flag.any():
        _, std_mc_z = mc_dropout_predict(
            model,
            data.X[morph_flag],
            data.line_idx[morph_flag].astype(np.int64),
            passes=MC_DROPOUT_PASSES,
        )
        std_mc = (std_mc_z * sig[data.line_idx[morph_flag]]).astype(np.float32) \
            if standardized else std_mc_z.astype(np.float32)
        std_all[morph_flag] = std_mc

    # --- Deterministic predictions for all samples ---
    y_det_all = eval_model(
        model, device,
        data.X, data.line_idx.astype(np.int64),
        mu, sig, standardized,
    )

    # --- Fallback decision ---
    # Samples are routed to fallback only if they are morphology-flagged AND
    # their uncertainty exceeds the calibrated threshold. NaN comparisons
    # correctly evaluate to False (non-morph samples are never routed to fallback).
    fallback = morph_flag & (std_all > float(UNC_PM_THRESH))

    # In this PoP the fallback uses pre-computed LineFit targets.
    # In a production pipeline, the fallback would invoke LineFit live for
    # the flagged (small) fraction of time steps.
    y_hybrid_all              = y_det_all.copy()
    y_hybrid_all[fallback]    = data.y_pm[fallback]

    frac_fallback = float(np.mean(fallback))
    print(f"  Fallback fraction: {100.0 * frac_fallback:.2f}%")

    # --- Uncertainty calibration metrics (validation morph-flagged samples) ---
    m_val_morph = m_val & morph_flag
    cov_dict: Dict[str, float] = {}
    if m_val_morph.any():
        cov_dict = uncertainty_coverage(
            yva_truth[ m_val[m_val_morph] ] if False else   # index alignment below
            data.y_truth_pm[m_val_morph],
            data.y_pm[m_val_morph],     # LineFit (proxy for truth in coverage check)
            std_all[m_val_morph],
        )
        # Re-compute using truth as reference
        cov_dict = uncertainty_coverage(
            data.y_truth_pm[m_val_morph],
            eval_model(model, device,
                       data.X[m_val_morph],
                       data.line_idx[m_val_morph].astype(np.int64),
                       mu, sig, standardized),
            std_all[m_val_morph],
        )
    print(f"  Uncertainty coverage (val morph): {cov_dict}")

    hybrid_summary = {
        "unc_pm_thresh":    float(UNC_PM_THRESH) if np.isfinite(UNC_PM_THRESH) else "inf (disabled)",
        "unc_thresh_rule":  unc_thresh_rule,
        "mc_passes":        MC_DROPOUT_PASSES,
        "frac_fallback":    float(frac_fallback),
        "frac_emulator":    float(1.0 - frac_fallback),
        "morphology_flag":  (
            f"splitcore_strict OR multipeak_scipy(height_rel={MULTIPEAK_HEIGHT_REL:.2f})"
        ),
        "morph_applied_to": f"stress line only (line_idx={LINE_STRESS})",
        "unc_coverage_val_morph": cov_dict,
        "note": (
            "Fallback uses pre-computed LineFit targets in this PoP. "
            "In production, fallback would invoke LineFit for flagged time steps only."
        ),
    }
    with open(os.path.join(OUTDIR, "hybrid_summary.json"), "w", encoding="utf-8") as f:
        json.dump(hybrid_summary, f, indent=2)

    # -------------------------------------------------------------------------
    # 5b. Estimate LineFit runtime on fallback time steps (single stress line)
    #
    #     This mirrors the approach used for the full-run LineFit reference
    #     timing in fig3_vtimeseries_calc.py: time a representative subset of
    #     steps and scale to the full fallback count. We use N=10 uniformly
    #     sampled fallback steps to keep the measurement to a few minutes while
    #     giving a stable per-step estimate.
    #
    #     Importantly, LineFit is invoked for the stress-case line ONLY (not
    #     all 10 lines), because in a production hybrid only the flagged line
    #     would be refitted at the fallback time steps. This gives the
    #     operationally correct fallback cost. The full-run reference time
    #     (_LINEFIT_FULL_TOTAL_S) covers all 10 lines × 300 steps; the hybrid
    #     total is therefore a conservative lower bound on the true speedup
    #     relative to running LineFit across all lines and all steps.
    # -------------------------------------------------------------------------
    N_FALLBACK_TIMING_SAMPLE = 10   # uniform sample; increase for a tighter estimate

    print("\nEstimating LineFit fallback runtime (single stress line, sample-based)...")

    fallback_sample_idx   = np.where(fallback)[0]
    fallback_timestep_idx = np.unique(data.time_idx[fallback_sample_idx].astype(int))
    n_fallback_ts         = int(fallback_timestep_idx.size)

    print(f"  Fallback time steps: {n_fallback_ts} / {nt} "
          f"({100.0 * n_fallback_ts / nt:.1f}%)")

    avg_lf_per_ts = _LINEFIT_FULL_TOTAL_S / nt   # per-step cost for all 10 lines

    if n_fallback_ts > 0:
        # Uniform sample across the fallback time steps for representativeness
        n_sample = min(N_FALLBACK_TIMING_SAMPLE, n_fallback_ts)
        sample_idx = fallback_timestep_idx[
            np.round(np.linspace(0, n_fallback_ts - 1, n_sample)).astype(int)
        ]
        sample_idx = np.unique(sample_idx)   # remove duplicates if n_fallback_ts < n_sample

        print(f"  Timing {len(sample_idx)} uniformly sampled fallback steps "
              f"(line {LINE_STRESS} only) — stdout suppressed, please wait...")

        _, t_sample_s = _run_linefit_on_timesteps(
            timestep_indices = sample_idx,
            spectra          = spectra,
            wl_nm            = wl_nm,
            lambda0_nm       = lambda0_nm,
            line_index       = LINE_STRESS,
        )

        fb_per_ts_single  = t_sample_s / len(sample_idx)   # per step, 1 line
        t_fallback_s      = fb_per_ts_single * n_fallback_ts  # scaled to all fallback steps

        print(f"  Measured: {t_sample_s:.2f} s for {len(sample_idx)} steps "
              f"(line {LINE_STRESS} only)")
        print(f"  Per-step (1 line): {fb_per_ts_single:.2f} s  "
              f"(full-run per-step avg for all 10 lines: {avg_lf_per_ts:.2f} s)")
        print(f"  Extrapolated fallback total ({n_fallback_ts} steps × "
              f"{fb_per_ts_single:.2f} s): {t_fallback_s:.1f} s")
        timing_method = (
            f"uniformly sampled {len(sample_idx)}/{n_fallback_ts} fallback steps, "
            f"stress line (line {LINE_STRESS}) only, scaled to all fallback steps"
        )
    else:
        t_fallback_s     = 0.0
        fb_per_ts_single = np.nan
        timing_method    = "no fallback steps"

    # Hybrid total: emulator over all 3000 samples + scaled fallback LineFit
    t_emulator_s     = float(t_full_emul_s)
    t_hybrid_s       = t_emulator_s + float(t_fallback_s)
    speedup_hybrid   = _LINEFIT_FULL_TOTAL_S / t_hybrid_s if t_hybrid_s > 0 else np.inf
    speedup_emulonly = _LINEFIT_FULL_TOTAL_S / max(t_emulator_s, 1e-9)

    print(f"\n{'='*60}")
    print(f"HYBRID TIMING SUMMARY")
    print(f"{'='*60}")
    print(f"  Full LineFit ref. (10 lines, 300 steps):  {_LINEFIT_FULL_TOTAL_S:>8.1f} s  [scaled from {_LF_TIMING_NFRAMES} steps]")
    print(f"  Emulator component (3000 samples):        {t_emulator_s:>8.4f} s")
    print(f"  Fallback (line {LINE_STRESS} only, {n_fallback_ts} steps, scaled): {float(t_fallback_s):>8.1f} s  [scaled from {len(sample_idx) if n_fallback_ts > 0 else 0} steps]")
    print(f"  Hybrid total:                             {t_hybrid_s:>8.1f} s")
    print(f"  Speedup hybrid vs full LineFit:           {speedup_hybrid:>8.1f}×")
    print(f"  Speedup emulator only (upper bound):      {speedup_emulonly:>8.0f}×")
    print(f"{'='*60}")

    timing = {
        "emulator_infer_pass_ms_valset":    float(infer_per_pass_ms),
        "emulator_infer_per_sample_us":     float(infer_per_sample_us),
        "emulator_full_dataset_ms":         float(1e3 * t_emulator_s),
        "n_emulator_samples":               int(data.X.shape[0] - len(fallback_sample_idx)),
        "fallback_timesteps_total":         int(n_fallback_ts),
        "fallback_timing_sample_size":      int(len(sample_idx)) if n_fallback_ts > 0 else 0,
        "fallback_timing_strategy":         "uniform sample across fallback time steps",
        "fallback_linefit_per_step_s":      float(fb_per_ts_single) if np.isfinite(fb_per_ts_single) else None,
        "fallback_linefit_extrapolated_s":  float(t_fallback_s),
        "fallback_timing_method":           timing_method,
        "fallback_line_fitted":             int(LINE_STRESS),
        "fallback_note": (
            "LineFit timed for stress-case line only (not all 10), matching production "
            "hybrid behaviour. Full-run reference covers all 10 lines × 300 steps "
            f"(scaled from {_LF_TIMING_NFRAMES} measured steps)."
        ),
        "hybrid_total_s":                   float(t_hybrid_s),
        "speedup_hybrid_vs_linefit":        float(speedup_hybrid),
        "linefit_full_total_s":             float(_LINEFIT_FULL_TOTAL_S),
        "linefit_avg_per_timestep_s":       float(avg_lf_per_ts),
        "speedup_emulator_only":            float(speedup_emulonly),
        "n_val_samples":                    int(Xva.shape[0]),
        "repeats":                          INFER_REPEATS,
        "warmup":                           TIMING_WARMUP,
        "mode":                             "torch.inference_mode",
        "impl":                             "torchscript_trace" if use_traced else "eager",
        "torch_threads":                    int(torch.get_num_threads()),
        "linefit_params_for_fallback": {
            "num_iterations":   _LF_NUM_ITERATIONS,
            "window_size":      _LF_WINDOW_SIZE,
            "smoothing_window": _LF_SMOOTHING_WINDOW,
            "line_window":      _LF_PER_LINE_WIN[LINE_STRESS],
            "line_min_hw":      _LF_PER_LINE_MIN_HW[LINE_STRESS],
            "line_max_hw":      _LF_PER_LINE_MAX_HW[LINE_STRESS],
            "reversal_line":    LINE_STRESS,
        },
    }
    with open(os.path.join(OUTDIR, "timing.json"), "w", encoding="utf-8") as f:
        json.dump(timing, f, indent=2)
    print(f"  Timing written to: {os.path.join(OUTDIR, 'timing.json')}")
    # -------------------------------------------------------------------------
    # 7. Stress-case line products
    # -------------------------------------------------------------------------
    m6    = data.line_idx == int(LINE_STRESS)
    order = np.argsort(data.time_idx[m6])

    y_truth6   = data.y_truth_pm[m6][order]
    y_lf6      = data.y_pm[m6][order]
    y_emul6    = y_det_all[m6][order]
    y_hyb6     = y_hybrid_all[m6][order]
    coarse6_nm = data.coarse_nm[m6][order]
    lam0_6     = float(lambda0_nm[LINE_STRESS])

    # Reconstruct absolute line-centre wavelengths
    def residual_to_velocity(coarse_nm_arr, resid_pm_arr, lam0):
        lamhat = coarse_nm_arr + resid_pm_arr.astype(np.float64) / 1e3
        return C_KMS * (lamhat - lam0) / lam0

    v_truth6   = residual_to_velocity(coarse6_nm, y_truth6,  lam0_6)
    v_linefit6 = residual_to_velocity(coarse6_nm, y_lf6,     lam0_6)
    v_emul6    = residual_to_velocity(coarse6_nm, y_emul6,   lam0_6)
    v_hyb6     = residual_to_velocity(coarse6_nm, y_hyb6,    lam0_6)

    np.savez(
        os.path.join(OUTDIR, "predictions_line6.npz"),
        time_s=time_s, coarse_nm=coarse6_nm,
        y_truth_pm=y_truth6, y_linefit_pm=y_lf6,
        y_emul_pm=y_emul6,   y_hybrid_pm=y_hyb6,
        v_truth_kms=v_truth6, v_linefit_kms=v_linefit6,
        v_emul_kms=v_emul6,   v_hybrid_kms=v_hyb6,
        UNC_PM_THRESH=float(UNC_PM_THRESH) if np.isfinite(UNC_PM_THRESH) else -1.0,
    )

    # -------------------------------------------------------------------------
    # 8. Validation diagnostics
    # -------------------------------------------------------------------------
    e_emul_truth_val   = (yhat_va - yva_truth).astype(np.float64)
    e_emul_linefit_val = (yhat_va - yva_linefit).astype(np.float64)
    y_hyb_va           = y_hybrid_all[m_val]
    e_hyb_truth_val    = (y_hyb_va - yva_truth).astype(np.float64)
    std_val            = std_all[m_val]
    morph_val          = morph_flag[m_val]
    fallback_val       = fallback[m_val]

    np.savez(
        os.path.join(OUTDIR, "val_diagnostics.npz"),
        yva_truth=yva_truth, yva_linefit=yva_linefit,
        yva_emul=yhat_va,    yva_hybrid=y_hyb_va,
        err_emul_truth=e_emul_truth_val,
        err_emul_linefit=e_emul_linefit_val,
        err_hyb_truth=e_hyb_truth_val,
        std_val=std_val, morph_val=morph_val, fallback_val=fallback_val,
        UNC_PM_THRESH=float(UNC_PM_THRESH) if np.isfinite(UNC_PM_THRESH) else -1.0,
    )

    # -------------------------------------------------------------------------
    # 9. Plots — essential figures only
    # -------------------------------------------------------------------------
    print("\nGenerating plots...")

    # (i) Validation error histogram and CDF
    plot_val_hist_cdf(PLOTDIR, np.abs(e_emul_truth_val),
                     np.abs(e_emul_linefit_val), np.abs(e_hyb_truth_val))

    # (ii) Fallback map — shows sparsity and temporal clustering
    plot_fallback_map(PLOTDIR, data, fallback)

    # (iii) RGWS preservation and error series (wave-diagnostic verification)
    plot_rgws_line6(PLOTDIR, time_s, v_truth6, v_linefit6, v_emul6, v_hyb6)

    # -------------------------------------------------------------------------
    # 10. Save consolidated metrics.json
    # -------------------------------------------------------------------------
    metrics_out = {
        "description": (
            "Proof-of-principle hybrid CNN emulator for LineFit acceleration. "
            "Emulator predicts line-centre residuals relative to coarse centre. "
            "LineFit outputs are used as training labels; truth used only for validation."
        ),
        "model":       "CNNLineEmbedWide (residual-to-coarse, validity-mask input, skip connection)",
        "fixed_config": FINAL_CFG,
        "fixed_seed":   FINAL_SEED,
        "picked_tag":   picked_tag,
        "splits": {
            "train_frac_time":        TRAIN_FRAC_TIME,
            "cal_frac_within_train":  CAL_FRAC_WITHIN_TRAIN,
            "val_frac_time":          1.0 - TRAIN_FRAC_TIME,
            "note": "Strictly time-ordered; no shuffling across time boundaries.",
        },
        "val_vs_linefit_pm":    m_vs_linefit,
        "val_vs_truth_pm":      m_vs_truth,
        "cal_vs_truth_pm":      result["cal_vs_truth"],
        "val_vs_truth_allpacks": result["val_vs_truth_allpacks"],
        "hybrid":               hybrid_summary,
        "timing": timing,
        "plot_flags": {"have_mpl": HAVE_MPL, "have_walsa": HAVE_WALSA},
    }
    with open(os.path.join(OUTDIR, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics_out, f, indent=2)

    # -------------------------------------------------------------------------
    # 11. Console summary
    # -------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("FINAL SUMMARY (fixed PoP configuration)")
    print("=" * 60)
    print(f"Seed: {FINAL_SEED}  |  Weights: '{picked_tag}'")
    print(f"VAL vs LineFit (pm): {m_vs_linefit}")
    print(f"VAL vs Truth   (pm): {m_vs_truth}")
    print(f"Hybrid fallback fraction: {100.0 * frac_fallback:.2f}%")
    print(f"UNC_PM_THRESH: {UNC_PM_THRESH:.4f} pm")
    print(f"Uncertainty coverage (val morph): {cov_dict}")
    print(f"Emulator (val-set): {infer_per_pass_ms:.2f} ms/pass  |  {infer_per_sample_us:.2f} µs/sample")
    print(f"Hybrid total runtime (est.): {t_hybrid_s:.1f} s  "
          f"(emulator: {t_emulator_s*1e3:.1f} ms + fallback line {LINE_STRESS}: {float(t_fallback_s):.1f} s)")
    print(f"Speedup vs full LineFit: {speedup_hybrid:.1f}×  "
          f"(emulator-only upper bound: {speedup_emulonly:.0f}×)")
    print(f"Fallback timing: {len(sample_idx) if n_fallback_ts > 0 else 0} steps measured, "
          f"scaled to {n_fallback_ts} total (stress line only)")
    print(f"Outputs: {OUTDIR}")
    print(f"Plots:   {PLOTDIR}")
    print("=" * 60)


if __name__ == "__main__":
    main()

Loading FITS testbed...
  Loaded: 300 time steps, 2400 wavelength channels, 10 lines.
Loading cached LineFit velocities...
Building ML dataset (residual-to-coarse formulation)...
  Dataset: 3000 samples, input shape (2, 57)

Training CNN emulator (seed=7000, epochs=160)...
  Early stopping at epoch 42 (patience=16).
  Training complete. Picked weights: 'best'
  VAL vs LineFit: {'mae_pm': 0.054793351254193115, 'rmse_pm': 0.09574818584326991, 'p95_abs_pm': 0.21100016534328445}
  VAL vs Truth:   {'mae_pm': 0.0795604286241966, 'rmse_pm': 0.11825232943530571, 'p95_abs_pm': 0.2448443204164505}

Benchmarking emulator inference...
  Emulator (val-set, 600 samples): 114.00 ms/pass, 190.00 µs/sample
  Emulator (full dataset, 3000 samples): 547.08 ms per pass

Applying hybrid gating to stress-case line...
  UNC_PM_THRESH = 0.1427 pm  (P95 of MC-dropout std on 24 morph-flagged CAL samples)
  Fallback fraction: 2.07%
  Uncertainty coverage (val morph): {'cov_1sigma': 0.43333333333333335, 'cov_2sigm

In [2]:
# Convenience: print outputs
with open("Files/PoP_CNN_emulator_outputs/metrics.json", "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
with open("Files/PoP_CNN_emulator_outputs/hybrid_summary.json", "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
with open("Files/PoP_CNN_emulator_outputs/timing.json", "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

{
  "description": "Proof-of-principle hybrid CNN emulator for LineFit acceleration. Emulator predicts line-centre residuals relative to coarse centre. LineFit outputs are used as training labels; truth used only for validation.",
  "model": "CNNLineEmbedWide (residual-to-coarse, validity-mask input, skip connection)",
  "fixed_config": {
    "lr": 0.001892149291938482,
    "weight_decay": 7.61522710343447e-05,
    "dropout_p": 0.04,
    "huber_beta": 0.05,
    "emb_dim": 16,
    "batch_size": 256,
    "sig_floor_pm": 0.05,
    "ema_decay": 0.995,
    "stress_diff_lambda": 0.0003
  },
  "fixed_seed": 7000,
  "picked_tag": "best",
  "splits": {
    "train_frac_time": 0.8,
    "cal_frac_within_train": 0.1,
    "val_frac_time": 0.19999999999999996,
    "note": "Strictly time-ordered; no shuffling across time boundaries."
  },
  "val_vs_linefit_pm": {
    "mae_pm": 0.054793351254193115,
    "rmse_pm": 0.09574818584326991,
    "p95_abs_pm": 0.21100016534328445
  },
  "val_vs_truth_pm": {
  